In [46]:
import pandas as pd 
import numpy as np
import sklearn 
import re 

In [47]:
df = pd.read_csv('online_retail_II.csv')

In [48]:
# Remove cancelled Order(Starts with C), remove negetive vlaues in price ans quantity features 
# And Make new feature Total_price 

df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] >= 0) & (df['Price'] >= 0) ]

df['Total_Price'] = df['Quantity'] * df['Price']

In [56]:
# there are 24k+ missing values in customer Id feature so before droping those value,
# we collect some info that might be help full in future

missing_revenue = df[df['Customer ID'].isna()]['Total_Price'].sum()
total_revenue = df['Total_Price'].sum()

print(f"Revanue from missing Customer ID: £ {missing_revenue:,.2f}")
print(f"Total Revanue: £ {total_revenue:,.2f}")
print(f"Revenue loss %: {missing_revenue/total_revenue*100:,.2f}%")

print(df[df['Customer ID'].isna()]['Country'].value_counts().head())

print(df[df['Customer ID'].isna()]['Invoice'].nunique())
    
df = df.dropna(subset=['Customer ID'])

Revanue from missing Customer ID: £ 3,229,165.39
Total Revanue: £ 20,972,594.57
Revenue loss %: 15.40%
Country
United Kingdom    235922
EIRE                1609
Hong Kong            358
Unspecified          231
France               128
Name: count, dtype: int64
4963


# RFM Engineering

In [89]:
# Calculating Recency, Frequency and Monetary from dataset by grouping them by CustomerID 
df = df.copy()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'],errors='coerce')

rafrance_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg(
    Recency = ('InvoiceDate',lambda x:(rafrance_date - x.max()).days),
    Frequency = ('Invoice','nunique'),
    Monetary = ('Total_Price','sum')
).reset_index()

# print(rfm.head())
# print(rfm.describe())